# PBA SE 2026 - Programa Alfabetiza Sergipe
## Relatório 3.1.3: Monitoramento da Aprendizagem e Rotina Pedagógica

**Autor:** Fabrício Camacho

**Objetivo:** Analisar o progresso de aprendizagem dos alfabetizandos e o cumprimento das metas operacionais estabelecidas no Termo de Referência relacionado ao resultado da **Atividade Diagnóstica de Entrada**.

**Data da Análise:** 02 de setembro de 2026.

## 1. Carregamento, Limpeza e Preparação dos Dados
Aqui vamos carregar os dados previamente importandos de arquivo JSON em formato de snapshot e transferidos para um arquivo `.csv`. Os dados foram consultados  através de snapshot, garantindo que as informações extraídas do JSON permaneçam os mesmos daqueles extraídos no dia da análise, evitando mudanças futuras dos dados que possam ocorrer no decorrer do programa.

### 1.1 Carregamento Bibliotecas


In [ ]:
# Importando bibliotecas
import pandas as pd
import re
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
import geopandas as gpd
from sklearn.cluster import KMeans
import math
from IPython.display import display
import textwrap
import json


### 1.2 Preenchendo requisitos para análise
Para seguir com a análise, é necessário preencher as variáveis abaixo:
- **ATIVIDADE**: a avaliação a ser analisada (1 a 5)
    - 0 - Diagnóstica de Entrada
    - 1 - Formativa 1
    - 2 - Formativa 2
    - 3 - Formativa 3
    - 4 - Formativa 4
    - 5 - Avaliação de Saída
- **DATA_EXTRACAO**: a data da extração dos dados no formato DDMMYYYY
- **DATA_REFERENCIA**: a data de referência para turmas de um contrato específico.
- **TURMAS_TESTE**: a lista de códigos de turmas-teste a serem retirados da análise

In [ ]:
# Preencher os seguintes valores:   
ATIVIDADE = 0 # Formativa a ser analisada
DATA_EXTRACAO = '02092026' # Data da extração dos dados no formato DDMMYYYY
DATA_REFERENCIA = ['2026-09-01']
TURMAS_TESTE = [
    'TURMA-P0000247-0001'
] 


### 1.3 Gerando Regras de Negócio estabelecidas para o PBA
Aqui vamos estabelecer os cálculos de pontuação das avaliações e do IPA previamente definidos para cada avaliação.

In [ ]:
# Modificações importantes para definir a formativa a ser analizada:
if ATIVIDADE == 0:
    NUM_FORMATIVA = 'diag_entr'
    PREFIXO_COLUNA = NUM_FORMATIVA
    NOME_AVALIACAO = 'Diagnóstica de Entrada'
    
elif ATIVIDADE == 5:
    NUM_FORMATIVA = 'diag_said'
    PREFIXO_COLUNA = NUM_FORMATIVA
    NOME_AVALIACAO = 'Atividade de Saída'
        
else:
    NUM_FORMATIVA = ATIVIDADE
    PREFIXO_COLUNA = f'forma_{NUM_FORMATIVA}' # Prefixo da coluna da base
    NOME_AVALIACAO = f'Atividade Formativa {NUM_FORMATIVA}' # Nome da avaliação para títulos e legendas

# Criando diretório para salvar gráficos para relatório
if ATIVIDADE == 0:
    dir_graficos = f'graficos_relatorio_3.1.3_{PREFIXO_COLUNA}'
    
elif ATIVIDADE == 5:
    dir_graficos = f'graficos_relatorio_3.2.3_{PREFIXO_COLUNA}'
    
elif ATIVIDADE == 1:
    dir_graficos = f'graficos_relatorio_2.1.3_{PREFIXO_COLUNA}'
        
elif ATIVIDADE == 2:
    dir_graficos = f'graficos_relatorio_2.2.3_{PREFIXO_COLUNA}'
    
else:
    dir_graficos = f'graficos_relatorio_2.3.3_{PREFIXO_COLUNA}'

if not os.path.exists(dir_graficos):
    os.makedirs(dir_graficos)
    
# Garantindo cores de gráficos de acordo com paleta estabelecida
cores_pba = [
    "#005088", "#FFBB00", "#00843D", "#333333", "#FF7F00", "#009BDB", 
    "#85A03A", "#A67C52", "#E5D9C5", "#5C6B73", "#333333"]

sns.set_palette(sns.color_palette(cores_pba)) # Definindo paleta de cores no ambiente seaborn

# Classificação de dificuldade das questões da Formativa em questão
classificacao_estabelecida_diag_entr = {
    'diag_entr_q1_pct': 'Fácil',
    'diag_entr_q2_pct': 'Fácil',
    'diag_entr_q3_pct': 'Intermediária',
    'diag_entr_q4_pct': 'Intermediária',
    'diag_entr_q5_pct': 'Difícil'
}

classificacao_estabelecida_forma_1 = {
    'forma_1_q1_pct': 'Fácil',
    'forma_1_q2_pct': 'Intermediária',
    'forma_1_q3_pct': 'Intermediária',
    'forma_1_q4_pct': 'Difícil',
    'forma_1_q5_pct': 'Difícil'
}

classificacao_estabelecida_forma_2 = {
    'forma_2_q1_pct': 'Fácil',
    'forma_2_q2_pct': 'Fácil',
    'forma_2_q3_pct': 'Intermediária',
    'forma_2_q4_pct': 'Intermediária',
    'forma_2_q5_pct': 'Difícil'
}

classificacao_estabelecida_forma_3 = {
    'forma_3_q1_pct': 'Fácil',
    'forma_3_q2_pct': 'Intermediária',
    'forma_3_q3_pct': 'Intermediária',
    'forma_3_q4_pct': 'Difícil',
    'forma_3_q5_pct': 'Difícil'
}

classificacao_estabelecida_forma_4 = {
    'forma_4_q1_pct': 'Fácil',
    'forma_4_q2_pct': 'Fácil',
    'forma_4_q3_pct': 'Intermediária',
    'forma_4_q4_pct': 'Intermediária',
    'forma_4_q5_pct': 'Difícil'
}

classificacao_estabelecida_diag_said = {
    'diag_said_q1_pct': 'Fácil',
    'diag_said_q2_pct': 'Fácil',
    'diag_said_q3_pct': 'Intermediária',
    'diag_said_q4_pct': 'Intermediária',
    'diag_said_q5_pct': 'Difícil'
}

# Definição de pontuação máxima para cada questão das Formativas
pontuacoes_maximas_diag_entr = {
    'diag_entr_q1': 5.0,
    'diag_entr_q2': 1.0,
    'diag_entr_q3': 6.0,
    'diag_entr_q4': 6.0,
    'diag_entr_q5': 6.0
}

pontuacoes_maximas_forma_1 = {
    'forma_1_q1': 6.0,
    'forma_1_q2': 8.0,
    'forma_1_q3': 2.0,
    'forma_1_q4': 4.0,
    'forma_1_q5': 12.0
}

pontuacoes_maximas_forma_2 = {
    'forma_2_q1': 12.0,
    'forma_2_q2': 3.0,
    'forma_2_q3': 3.0,
    'forma_2_q4': 6.0,
    'forma_2_q5': 8.0
}

pontuacoes_maximas_forma_3 = {
    'forma_3_q1': 4.0,
    'forma_3_q2': 6.0,
    'forma_3_q3': 1.0,
    'forma_3_q4': 16.0,
    'forma_3_q5': 5.0
}

pontuacoes_maximas_forma_4 = {
    'forma_4_q1': 4.0,
    'forma_4_q2': 14.0,
    'forma_4_q3': 4.0,
    'forma_4_q4': 6.0,
    'forma_4_q5': 4.0
}

pontuacoes_maximas_diag_said = {
    'diag_said_q1': 1.0,
    'diag_said_q2': 6.0,
    'diag_said_q3': 4.0,
    'diag_said_q4': 10.0,
    'diag_said_q5': 3.0
}

# Definindo cálculo para IPA para cada Formativa
def calcular_ipa_forma_1(df):
    # Usa estritamente as colunas 'forma_1' e trata dados vazios com 0
    df['pontuacao_ol_f1'] = df[['forma_1_q1', 'forma_1_q2', 'forma_1_q3', 'forma_1_q4', 'forma_1_q5']].fillna(0).sum(axis=1)
    df['pontuacao_pe_f1'] = df[['forma_1_q1', 'forma_1_q2', 'forma_1_q3', 'forma_1_q4', 'forma_1_q5']].fillna(0).sum(axis=1)
    df['pontuacao_al_f1'] = df[['forma_1_q2', 'forma_1_q5']].fillna(0).sum(axis=1)
    return df

def calcular_ipa_forma_2(df):
    # Usa estritamente as colunas 'forma_2' e trata dados vazios com 0
    df['pontuacao_ol_f2'] = df[['forma_2_q2', 'forma_2_q3', 'forma_2_q4']].fillna(0).sum(axis=1)
    df['pontuacao_pe_f2'] = df[['forma_2_q4', 'forma_2_q5']].fillna(0).sum(axis=1)
    df['pontuacao_al_f2'] = df[['forma_2_q1', 'forma_2_q2', 'forma_2_q3']].fillna(0).sum(axis=1)
    return df

def calcular_ipa_forma_3(df):
    # Usa estritamente as colunas 'forma_3' e trata dados vazios com 0
    df['pontuacao_ol_f3'] = df[['forma_3_q1', 'forma_3_q5']].fillna(0).sum(axis=1)
    df['pontuacao_pe_f3'] = df[['forma_3_q2', 'forma_3_q4', 'forma_3_q5']].fillna(0).sum(axis=1)
    df['pontuacao_al_f3'] = df[['forma_3_q1', 'forma_3_q3']].fillna(0).sum(axis=1)
    return df

def calcular_ipa_forma_4(df):
    # Usa estritamente as colunas 'forma_4' e trata dados vazios com 0
    df['pontuacao_ol_f4'] = df[['forma_4_q1', 'forma_4_q4', 'forma_4_q5']].fillna(0).sum(axis=1)
    df['pontuacao_pe_f4'] = df[['forma_4_q3', 'forma_4_q4', 'forma_4_q5']].fillna(0).sum(axis=1)
    df['pontuacao_al_f4'] = df[['forma_4_q1', 'forma_4_q2', 'forma_4_q3']].fillna(0).sum(axis=1)
    return df

# Definindo tetos máximos para cada formativa
tetos_forma_1 = {'ol': 32, 'pe': 32, 'al': 20}
tetos_forma_2 = {'ol': 12, 'pe': 14, 'al': 18}
tetos_forma_3 = {'ol': 9, 'pe': 27, 'al': 5}
tetos_forma_4 = {'ol': 14, 'pe': 14, 'al': 22}

# Definindo pesos das práticas de linguagem para cálculo de IPA
pesos = {'pe': 0.45, 'ol': 0.40, 'al': 0.15}

# Classificação Pedagógica utilizando os Thresholds Flexíveis para IPA
limites_ajustados = [0, 1.49, 2.49, 3.49, 4.01]
rotulos = ['Iniciante', 'Em desenvolvimento', 'Alfabetizado(a)', 'Alfabetização consolidada']

# Transformando a lista de datas de referência em data
DATA_REFERENCIA = pd.to_datetime(DATA_REFERENCIA)


### 1.4 Importando Datasets
Agora vamos criar os dataframes a partir dos dados fornecidos pelo sistema.

In [ ]:
# Importando arquivos separados
df_pedagogico = pd.read_csv(
    f'data_files/public_data/dados_pedagogicos_{DATA_EXTRACAO}.csv', encoding='utf-8-sig')
df_turmas = pd.read_csv(
    f'data_files/public_data/turmas_{DATA_EXTRACAO}.csv', encoding='utf-8-sig')

# Imprimindo dados
print("-"*100)
print("df_pedagogico:")
print(df_pedagogico.info())
display(df_pedagogico.head())

print("-"*100)
print("df_turmas:")
print(df_turmas.info())
display(df_turmas.head())


**Observação:** É possível notar que os dados apresentam algumas características que precisam ser alteradas. Seguem elas:
- **Nome das colunas:** colunas como `ds_turmas` e `turma_municipio` apresentam nomes que não são padronizados. Por esse motivo, precisamos alterar o nome das colunas para padronizar o nome dos dados;
- **Dados tipo "object" de algumas colunas:** As colunas *"status_alfabetizando", "diag_entr_result", "socio_entr_result", "forma_1_result", "forma_2_result", "forma_3_result", "forma_4_result", "diag_said_result"* e *"socio_said_result"* em `df_pedagógico` apresentam dados categóricos. Por esse motivo, precisamos alterar o tipo de dado principalemnte para poupar espaço em memória durante a análise;
- **Dados com data:** A coluna "dt_inicio_turma" em *"df_pedagogico", "Data início", "Data prevista de fim", "Data situação da turma"* e *"Data fim (máx.)"* em `df_turmas`; e *"DATA DE LANÇAMENTO"* em `df_relatorios_diagnostica` apresentam dados de data e hora e precisam ser transformados em tal formato.
- **Turmas que não fazem parte da análise:** A presente análise é referente a somente turmas que iniciaram dia 27/04/2026. Portanto retiraremos da análise todas as turmas que não iniciaram nessa data utilizando a coluna *"Data início"* em `df_turmas`.
- **Resultados das Atividades:** Os resultados das atividades apresentam texto longo. Vamos criar uma coluna a mais para cada resultado contendo somente o nível em que o alfabetizando foi classificado ou um texto mais resumido.
- **Turma Modelo:** A turma **TURMA-P0000247-0001** foi criada para teste e possui dados falsos. Todos os dados dela precisam ser retirados para limpar a análise;
- **Alunos Evadidos:** A base de dados contém alfabetizandos que evadiram e não apresentam resultados de atividades. Dessa maneira, alfabetizandos com status **"EVADIDO"** na coluna *"status_alfabetizando"* não serão considerados na análise.

### 1.5 Preparação de Dados
Aqui vamos fazer a limpeza e preparação dos dados da seguinte forma:
- Alterar nome das colunas `ds_turmas` e `turma_municipio` para `turma` e `municipio`, respectivamente para facilitar merges;
- Tranformar dados tipo *object* em dados categóricos ou de data de acordo com o observado anteriormente;
- Retirar turmas que não serão consideradas na análise;
- Resumir textos de resultados das atividades para melhor visualização;
- Criar colunas de resultados das atividades somente com o valor do Nível (N1, N2, N3 ou N4).


In [ ]:
# Trocando nome de coluna nos dfs para melhorar merges
def renomear_colunas_turmas(df):
    """
    Renomeia colunas do de dfs que contenham colunas com nome "ds_turmas" para "turma" para padronizar em todos os dfs.
    """
    # Criamos uma cópia para evitar o SettingWithCopyWarning
    df_resultado = df.copy()
    
    # Renomeia as colunas
    df_resultado = df_resultado.rename(columns={'ds_turma': 'turma', 'turma_municipio': 'municipio'})
    
    return df_resultado

# Transformando tipo de dados object em dados mais representativos (categorias e datas)
# Definindo colunas categóricas para df_pedagogico
cols_categoricas = [
    "status_alfabetizando", "diag_entr_result",
    "socio_entr_q1", "socio_entr_q2", "socio_entr_q3", "socio_entr_q4", 
    "socio_entr_q5", "socio_entr_q6", "socio_entr_q7",
    "forma_1_result", "forma_2_result", "forma_3_result", 
    "forma_4_result", "diag_said_result", 
    "socio_said_q1", "socio_said_q2", "socio_said_q3", "socio_said_q4", 
    "socio_said_q5", "socio_said_q6", "socio_said_q7", "socio_said_q8", "socio_said_q9"
]

def transformar_dados_pba(df):
    """
    Realiza a limpeza e transformação de tipos para o projeto Alfabetiza Sergipe.
    """
    # Transformação de Categóricos (df_pedagogico)
    for col in cols_categoricas:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    # Transformação de Datas (df_pedagogico)
    if 'dt_inicio_turma' in df.columns:
        df['dt_inicio_turma'] = pd.to_datetime(df['dt_inicio_turma'], dayfirst=True, errors='coerce')
        
    # Transformação de Datas (df_turmas)
    cols_datas_turmas = ["dt_inicio", "dt_fim", "data_situacao_turma", "dt_fim_previsao_f"]
    
    for col in cols_datas_turmas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')
        
    print("Transformação concluída com sucesso!")
    return df

# Aplicando a função
df_pedagogico, df_turmas = [transformar_dados_pba(renomear_colunas_turmas(df)) for df in [df_pedagogico, df_turmas]]

# Checando resultados
print(df_pedagogico.info())
print(df_turmas.info())


In [ ]:
# Mapeamento de variáveis categóricas para simplificar visualizações
mapeamento_textos = {
    'Demonstra motivação constante e desejo de continuar estudando': 'Demosntra motivação', 
    'Demonstra interesse, mas com oscilações': 'Demonstra interesse', 
    'Demonstra desmotivação ou desejo de interromper': 'Demonstra desmotivação', 
    'São frequentes e pontuais': 'Frequentes e pontuais', 
    'Frequência regular sem nenhuma falta': 'Frequência sem faltas', 
    'Frequência regular com algumas faltas': 'Frequência com faltas', 
    'Realiza com alguma ajuda': 'Realiza com ajuda', 
    'Realiza com autonomia na maioria das situações': 'Realiza com autonomia',
    'Depende de ajuda constante': 'Depende de ajuda', 
    'Tenta com apoio e incentivo': 'Tenta com apoio',
    'Tenta com iniciativa própria': 'Tenta com iniciativa',
    'Demonstra insegurança e evita tentar': 'Demonstra insegurança', 
    'Persiste e aceita o erro como parte da aprendizagem': 'Persiste e aceita o erro',
    'Persiste, mas demonstra frustração': 'Persiste com frustração',
    'Desiste facilmente': 'Desiste facilmente', 
    'Participa ativamente e coopera com os colegas': 'Participa e coopera',
    'Participa quando estimulado(a)': 'Participa quando estimulado(a)',
    'Evita interações': 'Evita interações', 
    'Reconhece claramente e relata usos práticos': 'Reconhece claramente',
    'Reconhece em algumas situações': 'Reconhece algumas vezes',
    'Não reconhece': 'Não reconhece', 
    'Boa motivação e interesse constante': 'Boa motivação e interesse',
    'Baixa motivação e risco de evasão': 'Baixa motivação',
    'Frequência regular e boa permanência': 'Frequênte e permanente',
    'Frequência regular com algumas faltas': 'Frequênte com faltas',
    'Frequência irregular e evasões significativas': 'Frequênte irregular',
    'Realizou com autonomia em várias situações': 'Realizou com autonomia',
    'Realizou com alguma ajuda': 'Realizou com ajuda',
    'Dependeu de ajuda constante': 'Dependeu de ajuda',
    'Demonstrou iniciativa e envolvimento': 'Demonstrou iniciativa',
    'Tentou com incentivo': 'Tentou com incentivo',
    'Demonstrou insegurança e resistência': 'Com insegurança e resistência',
    'Persistiu e compreendeu o erro como parte da aprendizagem': 'Persistiu e superou',
    'Persistiu com apoio': 'Persistiu com apoio',
    'Desistiu com facilidade': 'Desistiu facilmente',
    'Participou e cooperou ativamente com os colegas': 'Participou ativamente',
    'Interagiu quando estimulado(a)': 'Interagiu sob estimulação',
    'Apresentou pouca interação': 'Apresentou pouca interação',
    'Reconhecimento claro e frequente com usos práticos': 'Reconhecimento frequente',
    'Reconhecimento parcial': 'Reconhecimento parcial',
    'Pouco reconhecimento': 'Pouco reconhecimento',
    'Predominância nos níveis mais avançados': 'Níveis avançados',
    'Distribuição equilibrada entre níveis': 'Distribuição entre níveis',
    'Predominância nos níveis iniciais': 'Níveis iniciais',
    'Foram superados': 'Foram superados',
    'Tentou com incentivo': 'Tentou com incentivo',
    'Não foram superados': 'Não foram superados',
    'N1 | Não reconhece as letras do alfabeto': 'N1',
    'N2 | Lê e registra algumas letras com ajuda, sem relação entre fala e escrita': 'N2',
    'N3 | Reconhece algumas letras do alfabeto': 'N3',
    'N4 | Identifica o número de sílabas de palavras com apoio': 'N4', 
    'N1 | Atenção à escuta e reconhecimento inicial de letras': 'N1', 
    'N2 | Reconhece letras e sílabas com apoio': 'N2',
    'N3 | Identifica sílabas, sem relação letra–som': 'N3',
    'N4 | Reconhece sílabas, sem leitura de palavras': 'N4',
    'N1 | Identifica sílabas com apoio e reconhece palavras simples sem leitura autônoma': 'N1',
    'N2 | Reconhece o número de sílabas e identifica sílabas iguais com apoio': 'N2',
    'N3 | Decodifica palavras formadas por sílabas simples de forma lenta e silabada, com apoio': 'N3',
    'N4 | Lê palavras simples, ainda sem fluência e sem autonomia, demonstrando avanço na relação letra–som': 'N4',
    'N1 | Lê e escreve poucas palavras conhecidas, sempre com apoio': 'N1',
    'N2 | Lê e escreve palavras simples com autonomia inicial e possíveis erros': 'N2',
    'N3 | Lê e escreve palavras e frases simples, com possíveis erros ortográficos': 'N3',
    'N4 | Lê e escreve frases simples com maior segurança, ainda sem fluência leitora': 'N4',
    'N1 | Lê e escreve palavras e frases curtas com apoio e escreve o próprio nome com segurança': 'N1',
    'N2 | Lê e escreve frases curtas e pequenos textos, mesmo com erros ortográficos, e compreende informações explícitas': 'N2',
    'N3 | Lê, escreve e interpreta pequenos textos com compreensão básica e possíveis erros': 'N3',
    'N4 | Lê, escreve, compreende e interpreta pequenos textos, com possíveis erros ortográficos e fluência leitora inicial': 'N4',
    'N1 | Reconhece letras e sílabas com apoio; registra palavras e o nome; relação inicial fala–escrita': 'N1',
    'N2 | Reconhece e conta sílabas; relaciona letra e som; lê palavras com sílabas simples': 'N2',
    'N3 | Lê palavras e frases simples; escreve palavras e frases com sentido, mesmo com erros ortográficos': 'N3',
    'N4 | Lê pequenos textos com compreensão; escreve palavras, frases e pequenos textos; demonstra autonomia na leitura e escrita': 'N4'
}

# Criar a novas colunas resumidas para cada coluna categórica usando o .map()
for coluna in cols_categoricas:
    df_pedagogico[coluna + '_resum'] = df_pedagogico[coluna].map(mapeamento_textos)

# Transformar em tipo Categórico (ordenado) somente as colunas com níveis
# Isso é importante para que em gráficos o N1 venha antes do N2, etc.
niveis_ordenados = ['N1', 'N2', 'N3', 'N4']
colunas_niveis = ['diag_entr_result_resum', 'forma_1_result_resum', 'forma_2_result_resum', 'forma_3_result_resum', 'forma_4_result_resum', 'diag_said_result_resum']

for coluna in colunas_niveis:
    df_pedagogico[coluna] = pd.Categorical(
        df_pedagogico[coluna], 
        categories=niveis_ordenados, 
        ordered=True
    )

# Removemos colchetes, aspas duplas, aspas simples da coluna socio_entr_q9 e depois dividimos por vírgula
# 1. Criamos uma função inteligente para transformar o texto em uma lista real do Python
def extrair_lista_q9(valor):
    # Se o valor for vazio (NaN), retorna lista vazia
    if pd.isna(valor):
        return []
    
    # Se por acaso já for uma lista, ótimo, apenas retorna
    if isinstance(valor, list):
        return valor
    
    # Limpa espaços em branco nas bordas
    texto = str(valor).strip()
    
    if not texto or texto.lower() == 'nan':
        return []
    
    try:
        # O json.loads pega '["Falta de confian\u00e7a em si"]' e transforma perfeitamente na lista ['Falta de confiança em si']
        # Caso o CSV tenha duplicado aspas (ex: "[""...""]"), corrigimos antes
        texto = texto.replace('""', '"')
        lista = json.loads(texto)
        
        # Retorna a lista garantindo que não há itens vazios soltos
        return [str(item).strip() for item in lista if str(item).strip()]
    except:
        # PLANO B: Se o json falhar por algum caractere estranho, limpamos na mão
        limpo = texto.replace('[', '').replace(']', '').replace('"', '').replace("'", "")
        return [item.strip() for item in limpo.split(',') if item.strip()]
    
# 2. Aplicamos a nossa função na coluna para criar a coluna com as listas reais
df_pedagogico['socio_entr_q8_resum'] = df_pedagogico['socio_entr_q9'].apply(extrair_lista_q9)

# Verificando os tipos de dados e informações das colunas
print(df_pedagogico.info())
display(df_pedagogico.head())
display(df_pedagogico[['socio_entr_q9', 'socio_entr_q8_resum']].head())
print(df_pedagogico['socio_entr_q8_resum'].value_counts())


### 1.6 Limpeza e Organização de Dados
Aqui retiraremos alguns dados e adicionar informações para que a análise ocorra corretamente. Os seguintes dados serão retirados:
- Dados da turma **TURMA-P0000247-0001**, que foi criada para teste e não apresenta dados reais;
- Dados de alfabetizandos com status **EVADIDO**;
- Adicionar informação de município ao `df_pedagogico`, juntando com as informações de `df_turma`;
- Retirar valores vazios de alfabetizandos sem resultados


In [ ]:
# Verificando o tamanho antes da filtragem
print(f"Registros antes da filtragem: {len(df_pedagogico)}")

# Criando lista de turmas que devem ser excluidas do dataset
turmas_encerradas = df_turmas[df_turmas['situacao_turma'] == 'Encerrada']['turma'].unique()
turmas_fora_escopo = df_turmas[~df_turmas['dt_inicio'].isin(DATA_REFERENCIA)]['turma'].unique()

# 2. Aplicando os filtros
# Filtro: Turma diferente de 'TURMA-P0000247-0001' E Status diferente de 'EVADIDO'
def filtrar_turmas(df):
    """
    Filtra o DataFrame verificando se as colunas necessárias existem.
    """
    # Criamos uma cópia para evitar o SettingWithCopyWarning
    df_resultado = df.copy()
    
    # 1. Verificação da coluna 'turma'
    if 'turma' in df_resultado.columns:
        df_resultado = df_resultado[~df_resultado['turma'].isin(TURMAS_TESTE)]  # Filtra a turma de teste
    else:
        print("Aviso: Coluna 'turma' não encontrada em. Filtro de turma não aplicado.")
    
    # 2. Verificação da coluna 'status_alfabetizando'
    if 'status_alfabetizando' in df_resultado.columns:
        df_resultado = df_resultado[df_resultado['status_alfabetizando'] != 'EVADIDO']
    else:
        print("Aviso: Coluna 'status_alfabetizando' não encontrada. Filtro de evasão não aplicado.")
        
    # 3. Verificando e filtrando turmas fora do escopo e encerradas
    if 'turma' in df_resultado.columns:
        df_resultado = df_resultado[~df_resultado['turma'].isin(turmas_fora_escopo)] # Filtra turmas fora do escopo
        df_resultado = df_resultado[~df_resultado['turma'].isin(turmas_encerradas)] # Retira turmas encerradas do dataset
    else:
        print("Aviso: Coluna 'situacao_da_turma' não encontrada. Filtro de situação da turma não aplicado.")
    
    return df_resultado

# Aplicando as funções
df_pedagogico, df_turmas = [filtrar_turmas(df) for df in [df_pedagogico, df_turmas]]

# 3. Verificando o tamanho após a filtragem
print(f"Registros após a filtragem: {len(df_pedagogico)}")

# 4. Validando se a turma ainda existe no dataset
check_turma = any(turma in df_pedagogico['turma'].unique() for turma in TURMAS_TESTE)
print(f"As turmas-teste ainda estão presentes? {check_turma}")
print(f"Alfabetizandos cursando o programa atualmente: {len(df_pedagogico)}")


In [ ]:
# Adicionando informações de município aos dados pedagógicos de df_pedagogico
# Criando o DataFrame de referência apenas com as colunas necessárias
df_dados_turmas = df_turmas[['turma', 'ds_escola']].drop_duplicates()

# Realizando o merge
# Unimos pela coluna 'turma' (df_pedagógico) e 'turma' (df_turmas)
df_pedagogico = pd.merge(
    df_pedagogico, 
    df_dados_turmas, 
    on='turma',  
    how='left'
)

# Retirando linhas de alfabetizandos sem resultados
#df_pedagogico = df_pedagogico.dropna(subset=['diag_entr_result']).copy()
#df_pedagogico = df_pedagogico.dropna(subset=[f'{PREFIXO_COLUNA}_result']).copy()

# Verificação: conferir se há alunos sem município atribuído
nas_municipio = df_pedagogico['municipio'].isna().sum()
print(f"Alunos sem município após o join: {nas_municipio}")

if nas_municipio > 0:
    print("Aviso: Existem turmas no pedagógico que não constam no cadastro de turmas.")


## 2. Análise de Questões da Atividade Formativa em questão

A Avaliação Formativa é o nosso termómetro para medir o progresso da aprendizagem após intervenções pedagógicas. Nesta secção, temos os seguintes objetivos:
- Analisar a distribuição e a dificuldade das questões da Formativa em análise;
- Visualizar a distribuição das pontuações para cada questão;
- Realizar a classificação algorítmica das questões para verificar grupos a partir da dificuldade;
- Comparar as dificuldades pontuadas pela equipe pedagógica com às sinalizadas pelo algorítmo.

*Nota: Vamos filtrar os dados para garantir que contabilizamos apenas os alunos que efetivamente realizaram esta avaliação (removendo os valores nulos nesta coluna).*

### 2.1 Análise de Distribuição e Dificuldade das Questões Diagnósticas

Para entender quais habilidades foram melhor assimiladas e quais precisam de reforço, vamos analisar o desempenho individual por questão da avaliação diagnóstica de entrada. Como cada questão possui uma pontuação máxima diferente, primeiramente vamos normalizar os dados para uma escala percentual (0 a 100%).


In [ ]:
# Definindo as pontuações máximas por questão
variavel_pontuacao_maximas = f'pontuacoes_maximas_{PREFIXO_COLUNA}'
pontuacoes_maximas = globals()[variavel_pontuacao_maximas]

questoes = list(pontuacoes_maximas.keys())
colunas_pct = []

# Criando colunas percentuais para facilitar a comparação
for q, max_pts in pontuacoes_maximas.items():
    col_pct = f'{q}_pct'
    colunas_pct.append(col_pct)
    # Calcula o percentual e lida com eventuais valores nulos já existentes
    df_pedagogico[col_pct] = (df_pedagogico[q] / max_pts) * 100

# Visualizando as estatísticas descritivas percentuais
display(df_pedagogico[colunas_pct].describe().round(2))


#### 2.2 Distribuição das Pontuações
Através de Boxplots, podemos visualizar a dispersão das notas, as medianas e verificar se o comportamento da turma tende a gabaritar ou zerar as questões.

In [ ]:
plt.figure(figsize=(12, 6))

# Usando as cores oficiais do PBA para manter o padrão
sns.violinplot(data=df_pedagogico[colunas_pct], palette=cores_pba[:len(colunas_pct)])

plt.title(f'Distribuição Percentual de Pontuação por Questão - {NOME_AVALIACAO}', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('Pontuação (%)', fontsize=12)
plt.xlabel('Questões', fontsize=12)
plt.xticks(ticks=range(5), labels=['Questão 1', 'Questão 2', 'Questão 3', 'Questão 4', 'Questão 5'])
plt.ylim(-5, 105)

# Salvando gráfico
nome_grafico_2 = f'2_distribuicao_questoes_pontuacao_{PREFIXO_COLUNA}.png'
caminho_grafico_2 = os.path.join(dir_graficos, nome_grafico_2)

plt.savefig(caminho_grafico_2, dpi=300, bbox_inches='tight') # dpi=300 garante alta qualidade para relatórios/impressão
print(f"Gráfico salvo com sucesso em: {caminho_grafico_2}")
plt.show()


#### 2.3 Classificação Algorítmica das Questões
Utilizando a Teoria Clássica dos Testes associada ao **algoritmo K-Means**, agruparemos as questões em 3 níveis de dificuldade (Fácil, Intermediária e Difícil) baseando-nos puramente na média percentual de desempenho dos alfabetizandos.

In [ ]:
# 1. Calculando a média percentual de cada questão
medias_questoes = df_pedagogico[colunas_pct].mean().to_frame(name='media_percentual')

# 2. Aplicando o algoritmo K-Means para criar 3 clusters (Fácil, Intermediária, Difícil)
kmeans = KMeans(n_clusters=3, random_state=42)
medias_questoes['cluster'] = kmeans.fit_predict(medias_questoes[['media_percentual']])

# 3. Mapeando os clusters para os rótulos de dificuldade
# O cluster com a maior média será o Fácil, o do meio Intermediário, e o menor Difícil.
medias_ordenadas = medias_questoes.groupby('cluster')['media_percentual'].mean().sort_values()

# Dicionário dinâmico para mapear o id do cluster para o nome correto
mapa_dificuldade = {
    medias_ordenadas.index[0]: 'Difícil',
    medias_ordenadas.index[1]: 'Intermediária',
    medias_ordenadas.index[2]: 'Fácil'
}

medias_questoes['classificacao_algoritmo'] = medias_questoes['cluster'].map(mapa_dificuldade)
medias_questoes = medias_questoes.sort_values(by='media_percentual', ascending=False).round(2)

print("Classificação gerada pelo algoritmo K-Means:")
display(medias_questoes[['media_percentual', 'classificacao_algoritmo']])


#### 2.3.1 Descobrindo o Agrupamento Natural (Método do Cotovelo)
Antes de definirmos a classificação em 3 níveis (Fácil, Intermediário, Difícil), vamos utilizar o **Método do Cotovelo** para observar como as questões se agrupam naturalmente com base em suas médias percentuais. Calcularemos a inércia do K-Means para diferentes quantidades de clusters ($k$).

In [ ]:
# Importando a biblioteca caso ainda não tenha importado
inercia = []
# Como temos apenas 5 questões, vamos testar k de 1 até 4
valores_k = range(1, 5)

for k in valores_k:
    # O n_init=10 é adicionado para evitar warnings em versões mais recentes do scikit-learn
    kmeans_teste = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_teste.fit(medias_questoes[['media_percentual']])
    
    # inertia_ guarda a soma das distâncias quadradas dentro dos clusters
    inercia.append(kmeans_teste.inertia_)

# Plotando o gráfico do Método do Cotovelo
plt.figure(figsize=(10, 5))
plt.plot(valores_k, inercia, marker='o', linestyle='-', color='#005aa0', linewidth=2, markersize=8)

plt.title(f'Método do Cotovelo: Buscando o Agrupamento Natural das Questões da {NOME_AVALIACAO}', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Número de Grupos (k)', fontsize=12)
plt.ylabel('Inércia', fontsize=12)
plt.xticks(valores_k)
plt.grid(True, linestyle='--', alpha=0.6)

# Salvando o gráfico
nome_grafico_3 = f'3_metodo_cotovelo_questoes_{PREFIXO_COLUNA}.png'
caminho_grafico_3 = os.path.join(dir_graficos, nome_grafico_3)

plt.savefig(caminho_grafico_3, dpi=300, bbox_inches='tight')
print(f"Gráfico salvo com sucesso em: {caminho_grafico_2}")
plt.show()


In [ ]:
# 1. Pegamos os valores centrais (centroides) que o algoritmo encontrou e os ordenamos
centroides = kmeans.cluster_centers_.flatten()
centroides_ordenados = np.sort(centroides)

# 2. Calculamos o ponto médio entre os centroides para achar as fronteiras (notas de corte)
corte_dificil_inter = (centroides_ordenados[0] + centroides_ordenados[1]) / 2
corte_inter_facil = (centroides_ordenados[1] + centroides_ordenados[2]) / 2

print("--- REGRAS DE CLASSIFICAÇÃO DO ALGORITMO ---")
print(f"O algoritmo considerou os seguintes limites baseados na distância matemática:")
print(f"-> DIFÍCIL: Médias abaixo de {corte_dificil_inter:.2f}%")
print(f"-> INTERMEDIÁRIA: Médias entre {corte_dificil_inter:.2f}% e {corte_inter_facil:.2f}%")
print(f"-> FÁCIL: Médias acima de {corte_inter_facil:.2f}%")


#### 2.4 Comparação: Realidade Analítica vs. Parâmetro Estabelecido
Nesta etapa, cruzaremos os resultados obtidos pelo algoritmo (baseado no desempenho real dos alunos) com o grau de dificuldade originalmente planejado para a avaliação.

In [ ]:
# Adicionando a classificação estabelecida ao DataFrame
variavel_classificacao = f'classificacao_estabelecida_{PREFIXO_COLUNA}'
classificacao_estabelecida = globals()[variavel_classificacao]

# Aplicando o mapeamento
medias_questoes[variavel_classificacao] = medias_questoes.index.map(classificacao_estabelecida)

# Criando flag para saber se o algoritmo e o teste estão alinhados
medias_questoes['alinhamento'] = medias_questoes['classificacao_algoritmo'] == medias_questoes[variavel_classificacao]

# Formatando a tabela para exibição final
tabela_comparativa = medias_questoes[['media_percentual', variavel_classificacao, 'classificacao_algoritmo', 'alinhamento']].copy().sort_index()
tabela_comparativa.index = ['Questão 1', 'Questão 2', 'Questão 3', 'Questão 4', 'Questão 5'] # Ajuste na ordem se precisar
tabela_comparativa.columns = ['Média (%)', 'Dificuldade Planejada', 'Dificuldade Real (Algoritmo)', 'Alinhado?']

print("Comparativo Final de Dificuldade das Questões:")
display(tabela_comparativa)

# Opcional: Salvar a tabela comparativa em Excel para anexar ao relatório
caminho_comparativo = os.path.join(dir_graficos, f'3_comparativo_dificuldade_questoes_{PREFIXO_COLUNA}.xlsx')
tabela_comparativa.to_excel(caminho_comparativo)


## 3. Avaliação Pedagógica e Resultados do Diagnóstico Inicial

Este capítulo detalha os resultados obtidos através da aplicação da Formativa em questão. De acordo com o **Termo de Referência do PBA SE 2026**, o monitoramento da rotina de alfabetização deve partir de uma base sólida de dados que identifique o nível de adesão ao que foi ensinado em sala durante os estudos da última unidade.

A análise a seguir foca:
- Entender a distribuição geral de turmas e alfabetizandos entre os municípios;
- Entender a distribuição geral dos alfabetizandos pelos níveis **N1 a N4**, permitindo que a coordenação pedagógica ajuste as aulas e as formações de acordo com a realidade de cada turma. 
- Entender a distribuição dos alfabetizandos pelos níveis **N1 a N4** para cada município;
- Entender a realidade de alfabetizandos na Formativa em questão que tiraram niveis **N3 ou N4** durante a atividade diagnóstica, buscando checar o verdadeiro nível de proficiencia desses alfabetizandos ou entender se houve interferência do auxílio do alfabetizador na determinação do nível de alfabetização inicial.

### 3.1 Cobertura Territorial: Turmas e Alunos por Município

Para o monitoramento da rotina de alfabetização, é essencial cruzar o número de turmas ativas com o volume de alunos. Esta análise permite identificar a densidade do programa em cada localidade e verificar a conformidade com as metas de atendimento.


In [ ]:
# Configurações de estilo e paleta
sns.set_style("whitegrid", {'axes.grid' : False}) # Grid personalizado

# Consolidação de Dados por Município
# Criando um resumo por município a partir do df_turmas
resumo_municipios = df_turmas.groupby('municipio').agg(
    qtd_turmas=('turma', 'count'),
    total_alunos=('qtd_alfabetizandos', 'sum')
).sort_values(by='total_alunos', ascending=False).reset_index()

# Salvando arquivo
nome_do_arquivo_1 = 'turmas_e_alfabetizandos_por_municipio.xlsx'
caminho_excel_1 = os.path.join(dir_graficos, nome_do_arquivo_1)
resumo_municipios.to_excel(caminho_excel_1, index=False)

print(f"Planilha salva com sucesso em: {caminho_excel_1}")

# Checando turmas com Formativas registradas
alunos_com_formativa = df_pedagogico.dropna(subset=[f'{PREFIXO_COLUNA}_result'])
turmas_com_formativa = alunos_com_formativa['turma'].unique()
turmas_sem_formativa = list(set(df_pedagogico['turma'].unique()) - set(turmas_com_formativa))

# Imprimindo resultados
print(f'Quantidade de turmas no programa: {resumo_municipios["qtd_turmas"].sum()}')
print(f'Quantidade de municípios atendidos: {resumo_municipios.shape[0]}')
print(f'Quantidade de turmas com resultados de {NOME_AVALIACAO}: {len(turmas_com_formativa)}')
print(f'Turmas sem resultados da {NOME_AVALIACAO}: {len(turmas_sem_formativa)}')
print(f'Quantidade de municipios com turmas com resultados da {NOME_AVALIACAO}: {len(alunos_com_formativa["municipio"].unique())}')
print('-'*100)
print("Resumo de Distribuição por Município:")
display(resumo_municipios)
print('-'*100)
print(f'Turmas sem resultados da {NOME_AVALIACAO}:')
display(df_turmas[df_turmas['turma'].isin(turmas_sem_formativa)][['turma', 'alfabetizador', 'coordenador', 'municipio']].reset_index(drop=True))


In [ ]:
# Preparação dos dados para criação de gráfico de turmas e alfabetizandos(Garantindo que todos os municípios sejam incluídos)
resumo_completo = resumo_municipios.copy() # Usando o DF que criamos anteriormente
x_labels = resumo_completo['municipio'].values

# Configuração do gráfico
fig, ax1 = plt.subplots(figsize=(20, 10)) # Aumentamos a largura para caber todos

# --- EIXO 1: ALUNOS (BARRAS) ---
barras = ax1.bar(x_labels, resumo_completo['total_alunos'], color="#005088", label='Total de Alunos', alpha=0.8)
ax1.set_ylabel('Quantidade de Alunos', color="#005088", fontsize=14, fontweight='bold')
ax1.set_xlabel('Municípios', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', labelsize=14)
ax1.tick_params(axis='y', labelsize=14)

# Rótulos das barras (Alunos) - No topo das barras
ax1.bar_label(barras, padding=1, color="#005088", fontweight='bold', fontsize=11)

# Configuração do Eixo X para mostrar TODOS os nomes
ax1.set_xticks(range(len(x_labels)))
ax1.set_xticklabels(x_labels, rotation=90, fontsize=14)

# --- EIXO 2: TURMAS (LINHA) ---
ax2 = ax1.twinx()
linha = ax2.plot(x_labels, resumo_completo['qtd_turmas'], color="#FFBB00", marker='o', 
                 linewidth=2.5, markersize=6, label='Qtd. Turmas')
ax2.set_ylabel('Quantidade de Turmas', color="#FFBB00", fontsize=14, fontweight='bold')
ax2.tick_params(axis='y', labelsize=14)

# Rótulos da linha (Turmas) - ABAIXO do ponto e na COR #FFBB00
for i, valor in enumerate(resumo_completo['qtd_turmas']):
    ax2.text(i, valor - 0.4, str(int(valor)), color="#FFBB00", 
             ha='center', va='top', fontweight='bold', fontsize=11)

# Unificando as legendas
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc='upper right', bbox_to_anchor=(1, 1.1), fontsize=14)

plt.title('Alfabetizandos e Turmas por Município', fontsize=20, pad=40, fontweight='bold')
plt.tight_layout()
sns.despine(right=False)

# Salvando o gráfico em dir_graficos
nome_grafico_4 = '4_numero_de_turmas_e_alunos_por_municipio.png'
caminho_grafico_4 = os.path.join(dir_graficos, nome_grafico_4)

plt.savefig(caminho_grafico_4, dpi=300, bbox_inches='tight') # dpi=300 garante alta qualidade para relatórios/impressão
print(f"Gráfico salvo com sucesso em: {caminho_grafico_4}")
plt.show()


### 3.2 Níveis adesão ao conteúdo estudado (Formativa)

Para fins deste relatório, a classificação segue os critérios estabelecidos pela FGV DGPE para cada atividade aplicada:

**Atividade Diagnóstica de Entrada**

* **N1:** Não reconhece as letras do alfabeto.
* **N2:** Lê e registra algumas letras com ajuda sem relação entre fala e escrita.
* **N3:** Reconhece algumas letras do alfabeto.
* **N4:** Identifica o número de sílabas de palavras com apoio.

**Atividade Formativa 1**

* **N1:** Participa das atividades e demonstra atenção à escuta de palavras, sem reconhecer letras ou sílabas.
* **N2:** Reconhece algumas letras do alfabeto e identifica o número de sílabas de palavras com apoio.
* **N3:** Reconhece letras com mais segurança e identifica sílabas em palavras, relacionando letra e som.
* **N4:** Reconhece letras e sílabas em palavras e identifica a quantidade de sílabas. Está em processo de leitura e escrita de palavras.

**Atividade Formativa 2**

* **N1:** Identifica sílabas com apoio e reconhece palavras sem leitura autônoma.
* **N2:** Reconhece o número de sílabas e identifica sílabas com apoio.
* **N3:** Decodifica palavras formadas por sílabas de forma lenta, com apoio.
* **N4:** Faz a leitura de palavras, ainda sem fluência e sem autonomia, demonstrando avanço na relação letra–som.

**Atividade Formativa 3**

* **N1:** Participa das atividades, sem consolidar a leitura e a escrita de palavras.
* **N2:** Lê e escreve palavras com autonomia e possíveis erros.
* **N3:** Lê e escreve palavras e frases com possíveis erros ortográficos.
* **N4:** Lê e escreve frases com maior segurança, ainda sem fluência leitora e com erros ortográficos.

**Atividade Formativa 4**

* **N1:** Participa das atividades, sem consolidar a leitura e a escrita de palavras e frases.
* **N2:** Lê e escreve frases curtas e pequenos textos, mesmo com erros ortográficos, e compreende informações explícitas.
* **N3:** Lê, escreve e interpreta pequenos textos com compreensão básica  e erros ortográficos.
* **N4:** Lê, escreve, compreende e interpreta pequenos textos, com erros ortográficos e fluência leitora inicial.

**Atividade Diagnóstica de Saída**

* **N1:** Participa das atividades, sem consolidar o reconhecimento de letras e sílabas nem o registro de palavras e do próprio nome.
* **N2:** Reconhece e conta sílabas, relaciona letras e sons e lê palavras.
* **N3:** Lê e escreve palavras e frases com sentido, ainda que apresente erros ortográficos.
* **N4:** Lê e escreve palavras, frases e pequenos textos com compreensão. Demonstra autonomia na leitura e escrita.


In [ ]:
# 1. Filtrar apenas os alunos que realizaram a Atividade
df_formativa = df_pedagogico.dropna(subset=[f'{PREFIXO_COLUNA}_result_resum']).copy()

print(f"Total de alunos que realizaram a {NOME_AVALIACAO}: {df_formativa.shape[0]}")

# 2. Configurar e criar o gráfico
plt.figure(figsize=(10, 6))
ordem_fixa = ['N4', 'N3', 'N2', 'N1']  # Ordem desejada dos níveis

sns.countplot(
    y=f'{PREFIXO_COLUNA}_result_resum', 
    data=df_formativa, 
    order=ordem_fixa, 
    palette=cores_pba,
    hue=f'{PREFIXO_COLUNA}_result_resum', # Evitar warning da paleta
    legend=False
)

plt.title(f'Distribuição Geral de Resultados - {NOME_AVALIACAO}', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Número de Alfabetizandos', fontsize=12)
plt.ylabel(f'Nível alcançado na {NOME_AVALIACAO}', fontsize=12)

# Adicionar os valores exatos em cada barra para ficar mais profissional
ax = plt.gca()
for p in ax.patches:
    ax.annotate(f'{int(p.get_width())}', 
                (p.get_width(), p.get_y() + p.get_height() / 2.), 
                ha='left', va='center', 
                xytext=(5, 0), textcoords='offset points', fontsize=10)
    
# Salvando o gráfico
nome_grafico_5 = f'5_distribuicao_geral_{PREFIXO_COLUNA.lower().replace(" ", "_")}.png'
caminho_grafico_5 = os.path.join(dir_graficos, nome_grafico_5)

plt.savefig(caminho_grafico_5, dpi=300, bbox_inches='tight')
print(f"Gráfico salvo com sucesso em: {caminho_grafico_5}")

plt.tight_layout()
plt.show()


### 3.3 Resultados da Formativa por Município (Geração de Gráficos Individuais)

Para subsidiar as equipes técnicas locais, automatizamos a geração de gráficos de desempenho para cada um dos municípios parceiros. 

**Processo de Automação:**
1. O código identifica todos os municípios únicos na base de dados.
2. Filtra os alfabetizandos ativos de cada localidade.
3. Gera um gráfico horizontal respeitando a identidade visual do PBA.
4. Salva o arquivo `.png` no diretório de saída (`dir_graficos`) para inclusão automática nos relatórios municipais.

In [ ]:
df_formativa['socio_entr_q9'].value_counts(normalize=True)

In [ ]:
# Lista de municípios únicos
municipios = sorted(df_pedagogico['municipio'].astype(str).unique())

# Gerando loop para criar um gráfico para cada município
print(f"Iniciando a geração de gráficos padronizados para {len(municipios)} municípios...")

for muni in municipios:
    df_muni = df_formativa[df_formativa['municipio'] == muni]
    
    # Agrupar dados
    resultado_muni = df_muni.groupby([f'{PREFIXO_COLUNA}_result_resum'], observed=False)['cpf'].count().reset_index()
    resultado_muni.columns = [f'resultado_{PREFIXO_COLUNA}', 'numero_alfabetizandos']
    
    # Mantemos o figsize similar ao seu gráfico geral para proporção
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # --- AJUSTE DE ESPESSURA ---
    sns.barplot(
        data=resultado_muni, 
        x='numero_alfabetizandos', 
        y=f'resultado_{PREFIXO_COLUNA}', 
        palette=cores_pba,
        hue=f'resultado_{PREFIXO_COLUNA}', # Colorir a barra com base no resultado
        order=ordem_fixa, # ESSENCIAL: Garante que o eixo Y tenha sempre 4 posições
        height=0.8,       # FIXA A ESPESSURA: 0.8 é a largura padrão do Seaborn
        ax=ax
    )
    
    # Adicionar rótulos (Loop em todos os containers)
    for container in ax.containers:
        ax.bar_label(container, padding=10, fontweight='bold', color='#333333')
    
    # Títulos e Estética (Seguindo seu padrão)
    ax.set_title(f'Resultados da {NOME_AVALIACAO} em {muni}', fontsize=14, pad=15, fontweight='bold', color="#005088")
    ax.set_xlabel('Quantidade de Alunos', fontsize=10)
    ax.set_ylabel(f'Nível alcançado na {NOME_AVALIACAO}', fontsize=10)
    
    # Ajuste de margem (1.4 garante espaço para o rótulo à direita)
    max_val = resultado_muni['numero_alfabetizandos'].max()
    ax.set_xlim(0, (max_val * 1.4) if max_val > 0 else 10)
    
    sns.despine()
    plt.tight_layout()
    
    # Salvando o arquivos no diretório dir_graficos
    nome_arquivo = f"{PREFIXO_COLUNA.lower().replace(' ', '_')}_{muni.lower().replace(' ', '_')}.png"
    caminho_salvamento = os.path.join(dir_graficos, nome_arquivo)
    
    plt.savefig(caminho_salvamento, dpi=300, bbox_inches='tight')
    plt.close(fig)
    
# Criando resultados por municípios
tabela_resultados_municipios = pd.pivot_table(
    df_formativa,
    index='municipio',           # O que vai ficar nas linhas
    columns=f'{PREFIXO_COLUNA}_result_resum',    # O que vai ficar nas colunas (N1, N2, N3, N4)
    values='cpf',                # O que vai ser contado (alunos)
    aggfunc='nunique',              # A operação: contar a quantidade de linhas (alunos)
    fill_value=0                 # Preencher com 0 onde não houver alunos (em vez de NaN)
)

# Adicionar uma coluna de 'Total' para enriquecer a tabela
tabela_resultados_municipios['Total'] = tabela_resultados_municipios.sum(axis=1)

# Ordenar a tabela pela quantidade total de alunos no município (do maior para o menor)
tabela_resultados_municipios = tabela_resultados_municipios.sort_values(by='Total', ascending=False)

# Salvando o arquivo no diretório dir_graficos
nome_do_arquivo_3 = f'{PREFIXO_COLUNA.lower().replace(" ", "_")}_resultados_por_municipio.xlsx'
caminho_excel_3 = os.path.join(dir_graficos, nome_do_arquivo_3)
tabela_resultados_municipios.to_excel(caminho_excel_3, index=True)

print(f"Sucesso! Gráficos com espessura padronizada salvos em: {dir_graficos}")
display(tabela_resultados_municipios.head())


### 3.4 Transição de Aprendizagem: O Desempenho dos Estudantes N3 e N4

Uma questão central para a nossa estratégia pedagógica é perceber o que acontece com os alunos que iniciam o programa com um nível mais avançado (Níveis N3 ou N4 na Avaliação Diagnóstica). Será que a Formativa em questão demonstrou que eles evoluíram, mantiveram o nível ou regrediram?

Abaixo, isolamos o grupo de alunos "N3 ou N4" da avaliação de entrada e cruzamos com os resultados que obtiveram agora na Formativa 1 através de uma **Matriz de Transição**.

In [ ]:
# 1. Filtrar alunos que tiraram N3 ou N4 na diagnóstica
n3_n4_diagnostica = df_pedagogico[df_pedagogico['diag_entr_result_resum'].str.contains('N3|N4', na=False, regex=True)].copy()

# 2. Remover quem não fez a formativa 1 dentro deste grupo
n3_n4_diagnostica = n3_n4_diagnostica.dropna(subset=['forma_1_result_resum'])

print(f"Total de alunos (N3 ou N4 na entrada) que realizaram a Formativa 1: {n3_n4_diagnostica.shape[0]}\n")

# 3. Plotar o desempenho específico destes alunos na Formativa 1
plt.figure(figsize=(10, 5))
sns.countplot(
    y='forma_1_result_resum', 
    data=n3_n4_diagnostica, 
    order=ordem_fixa, 
    palette=cores_pba,
    hue='forma_1_result_resum',
    legend=False
)
plt.title(f'Desempenho na {NOME_AVALIACAO} (Apenas alunos N3 e N4 na Entrada)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Número de Alfabetizandos')
plt.ylabel(f'Nível - {NOME_AVALIACAO}')

# Salvando o gráfico
nome_grafico_6 = f'6_desempenho_na_{PREFIXO_COLUNA.lower().replace(" ", "_")}_apenas_alunos_n3_e_n4_na_entrada.png'
caminho_grafico_6 = os.path.join(dir_graficos, nome_grafico_6)

plt.savefig(caminho_grafico_6, dpi=300, bbox_inches='tight')
print(f"Gráfico salvo com sucesso em: {caminho_grafico_6}")

plt.tight_layout()
plt.show()

# 4. Gerar a Matriz de Transição
print(f"\n--- Matriz de Transição (Quantidade de Alunos): Diagnóstica de Entrada -> {NOME_AVALIACAO} ---")
matriz_transicao = pd.crosstab(
    n3_n4_diagnostica['diag_entr_result'], 
    n3_n4_diagnostica[f'{PREFIXO_COLUNA}_result_resum'], 
    margins=True, 
    margins_name="Total Geral"
)

# Estilizando a tabela para destacar os números com cores
cm = sns.light_palette("purple", as_cmap=True)
tabela_estilizada = matriz_transicao.style.background_gradient(cmap=cm)

display(tabela_estilizada)


### 3.5 Cálculo do Índice de Progressão de Aprendizagem (IPA) - Formativa

Com os relatórios de notas consolidados, vamos calcular o **IPA** para a Formativa em questão. Diferente de uma média simples, o IPA isola o desempenho do alfabetizando em cada prática de linguagem e aplica a regra de progressão contínua na Zona de Desenvolvimento Proximal (ZPD).

A fórmula fundamental de consolidação do sub-nível é:
$$IPA_{i,p} = \max\left(IPA_{i-1,p}, i \times S_{i,p}\right)$$

**Passos executados neste bloco:**
1. **Mapeamento de Questões:** Somamos os acertos agrupando pelas práticas (Oralidade e Leitura, Produção Escrita e Análise Linguística) de acordo com os descritores avaliados em cada questão da F1.
2. **Cálculo do Percentual ($S$):** Dividimos a soma obtida pelo teto máximo de cada prática.
3. **Ponderação Global:** Consolidamos o IPA Global atribuindo **45%** para Escrita, **40%** para Leitura e **15%** para Análise Linguística.
4. **Classificação:** Categorizamos o resultado em 4 níveis pedagógicos com bandas flexíveis para absorver pequenas margens de erro.

> **Observação:** O IPA só será calculado para as avaliações Formativas (1 a 4).

In [ ]:
# Calculando IPA de qualquer Formativa (dinâmico)
def calcular_ipa_geral(df, num_formativa=None):
    
    # Condição para não calcular IPA para Diagnóstica de Entrada ou Atividade de Saída
    if num_formativa is None:
        num_formativa = NUM_FORMATIVA
        
    if NUM_FORMATIVA in ['diag_entr', 'diag_said']:
        print(f"Não é possível calcular o IPA para dados da {NOME_AVALIACAO}.")
        return df  # <- O return faz a função parar aqui    
       
    prefixo_coluna = f'forma_{num_formativa}'
    
    # 1. Calculando as pontuações brutas da Formativa em análise
    nome_funcao_calculo_ipa = f'calcular_ipa_{prefixo_coluna}'
    df = df.pipe(globals()[nome_funcao_calculo_ipa])
    
    # 2. Definir os Parâmetros (Tetos Máximos) da formativa em análise
    tetos_formativa = globals()[f'tetos_{prefixo_coluna}']
    
    # --- AJUSTE DINÂMICO DA BASE ---
    if num_formativa > 1:
        num_formativa_anterior = num_formativa - 1
        
        # O prefixo de onde vamos puxar o nível base para não deixar a nota cair
        base_ol = f'ipa_ol_f{num_formativa_anterior}'
        base_pe = f'ipa_pe_f{num_formativa_anterior}'
        base_al = f'ipa_al_f{num_formativa_anterior}'
        
        # Se os IPAs da formativa anterior ainda não existirem na tabela, chamamos esta MESMA função para calculá-los antes de prosseguir!
        if base_ol not in df.columns:
            df = calcular_ipa_geral(df, num_formativa=num_formativa_anterior)
            
    else:
        # Se for a Formativa 1, inicializa a diagnóstica caso não exista
        for pratica in ['ol', 'pe', 'al']:
            col_name = f'ipa_diagnostica_{pratica}'
            if col_name not in df.columns:
                df[col_name] = 1.0 # Nível base
        
        base_ol = 'ipa_diagnostica_ol'
        base_pe = 'ipa_diagnostica_pe'
        base_al = 'ipa_diagnostica_al'
    # ------------------------------------------------
    
    # 3. Calcular o Percentual de Acerto (S) na Formativa Atual
    df[f's_ol_f{num_formativa}'] = df[f'pontuacao_ol_f{num_formativa}'] / tetos_formativa['ol']
    df[f's_pe_f{num_formativa}'] = df[f'pontuacao_pe_f{num_formativa}'] / tetos_formativa['pe']
    df[f's_al_f{num_formativa}'] = df[f'pontuacao_al_f{num_formativa}'] / tetos_formativa['al']

    # 4. Calcular o Sub-nível de Cada Prática (IPA por Prática) com base na linha de base escolhida
    df[f'ipa_ol_f{num_formativa}'] = np.fmax(df[base_ol], num_formativa * df[f's_ol_f{num_formativa}'])
    df[f'ipa_pe_f{num_formativa}'] = np.fmax(df[base_pe], num_formativa * df[f's_pe_f{num_formativa}'])
    df[f'ipa_al_f{num_formativa}'] = np.fmax(df[base_al], num_formativa * df[f's_al_f{num_formativa}'])

    # 5. Calcular o IPA Global (Índice Composto)
    df[f'ipa_global_f{num_formativa}'] = (
        (pesos['pe'] * df[f'ipa_pe_f{num_formativa}']) + 
        (pesos['ol'] * df[f'ipa_ol_f{num_formativa}']) + 
        (pesos['al'] * df[f'ipa_al_f{num_formativa}'])
    ).round(2)

    # 6. Classificação Pedagógica
    df[f'classificacao_ipa_f{num_formativa}'] = pd.cut(
        df[f'ipa_global_f{num_formativa}'], 
        bins=limites_ajustados, 
        labels=rotulos, 
        right=True
    )

    # Ordenando categorias
    df[f'classificacao_ipa_f{num_formativa}'] = pd.Categorical(
        df[f'classificacao_ipa_f{num_formativa}'], 
        categories=rotulos, 
        ordered=True
    )

    # 7. Visualização Final
    colunas_visualizacao = [
        f'pontuacao_ol_f{num_formativa}', 
        f'pontuacao_pe_f{num_formativa}', 
        f'pontuacao_al_f{num_formativa}', 
        f'ipa_global_f{num_formativa}',
        f'classificacao_ipa_f{num_formativa}'
    ]
    
    print(f"\n--- Amostra de Alunos: Formativa {num_formativa} ---")
    display(df[colunas_visualizacao].head(10))
    print(f"\n--- Total de Alunos por Classificação (Formativa {num_formativa}) ---")
    
    # Criamos um DataFrame temporário para juntar a contagem e a porcentagem
    contagem = df[f'classificacao_ipa_f{num_formativa}'].value_counts()
    porcentagem = df[f'classificacao_ipa_f{num_formativa}'].value_counts(normalize=True) * 100
    
    resumo_classificacao = pd.DataFrame({
        'Quantidade': contagem,
        'Percentual (%)': porcentagem.round(2)
    })
    
    display(resumo_classificacao)
    print("-" * 50)
        
    return df

# Aplicando a função para calcular o IPA geral da Formativa
df_formativa = calcular_ipa_geral(df_formativa)


In [ ]:
# Criando condição para não gerar gráfico de IPA para Diagnóstica de Entrada ou Atividade de Saída
if NUM_FORMATIVA in ['diag_entr', 'diag_said']:
    print(f"Não é possível gerar gráfico de IPA para {NOME_AVALIACAO}.")
    
else:
    # Criando ordem mais adequada de exposição de classificação de IPA no gráfico
    ordem_descricao_ipa = ['Alfabetização consolidada', 'Alfabetizado(a)', 'Em desenvolvimento', 'Iniciante']

    # Plotar o desempenho específico destes alunos na Formativa 1
    plt.figure(figsize=(10, 5))
    sns.countplot(
        y=f'classificacao_ipa_f{NUM_FORMATIVA}', 
        data=df_formativa, 
        order=ordem_descricao_ipa,
        palette=cores_pba,
        hue=f'classificacao_ipa_f{NUM_FORMATIVA}',
        height=0.8,       # FIXA A ESPESSURA: 0.8 é a largura padrão do Seaborn
        legend=False
    )
    plt.title(f'Índice de Progresso de Aprendizagem (IPA) para resultados da Formativa {NUM_FORMATIVA} - Geral', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Número de Alfabetizandos')
    plt.ylabel('Progresso de Aprendizagem')

    # Salvando o gráfico
    nome_grafico_7 = f'7_ipa_{PREFIXO_COLUNA}_geral.png'
    caminho_grafico_7 = os.path.join(dir_graficos, nome_grafico_7)

    plt.savefig(caminho_grafico_7, dpi=300, bbox_inches='tight')
    print(f"Gráfico salvo com sucesso em: {caminho_grafico_7}")

    plt.tight_layout()
    plt.show()


In [ ]:
# Criando condição para não gerar gráfico de IPA para Diagnóstica de Entrada ou Atividade de Saída
if NUM_FORMATIVA in ['diag_entr', 'diag_said']:
    print(f"Não é possível gerar gráfico de IPA para {NOME_AVALIACAO}.")
    
else:
    # Gerando loop para criar um gráficos de IPA para cada município
    print(f"Iniciando a geração de gráficos padronizados de IPA para {len(municipios)} municípios...")
    for muni in municipios:
        df_muni = df_formativa[df_formativa['municipio'] == muni]
        
        # Agrupar dados
        ipa_forma_municipios = df_muni.groupby(['municipio', f'classificacao_ipa_f{NUM_FORMATIVA}'], observed=False)['cpf'].nunique().reset_index()
        ipa_forma_municipios.columns = ['municipio', f'classificacao_ipa_f{NUM_FORMATIVA}', f'numero_alfabetizandos_f{NUM_FORMATIVA}']
        
        # Mantemos o figsize similar ao seu gráfico geral para proporção
        fig, ax = plt.subplots(figsize=(10, 5))
        
        # --- AJUSTE DE ESPESSURA ---
        sns.barplot(
            data=ipa_forma_municipios, 
            x=f'numero_alfabetizandos_f{NUM_FORMATIVA}', 
            y=f'classificacao_ipa_f{NUM_FORMATIVA}', 
            palette=cores_pba,
            hue=f'classificacao_ipa_f{NUM_FORMATIVA}', # Colorir a barra com base na classificação IPA
            order=ordem_descricao_ipa, # ESSENCIAL: Garante que o eixo Y tenha sempre 4 posições
            height=0.8,       # FIXA A ESPESSURA: 0.8 é a largura padrão do Seaborn
            ax=ax
        )
        
        # Adicionar rótulos (Loop em todos os containers)
        for container in ax.containers:
            ax.bar_label(container, padding=10, fontweight='bold', color='#333333')
        
        # Títulos e Estética (Seguindo seu padrão)
        ax.set_title(f'IPA para Turmas em {muni}', fontsize=14, pad=15, fontweight='bold', color="#005088")
        ax.set_xlabel('Quantidade de Alunos', fontsize=10)
        ax.set_ylabel('Progresso de Aprendizagem', fontsize=10)
        
        # Ajuste de margem (1.4 garante espaço para o rótulo à direita)
        max_val = ipa_forma_municipios[f'numero_alfabetizandos_f{NUM_FORMATIVA}'].max()
        ax.set_xlim(0, (max_val * 1.4) if max_val > 0 else 10)
        
        sns.despine()
        plt.tight_layout()
        
        # Salvando o arquivos no diretório dir_graficos
        nome_arquivo = f"IPA_{PREFIXO_COLUNA}_{muni.lower().replace(' ', '_')}.png"
        caminho_salvamento = os.path.join(dir_graficos, nome_arquivo)
        
        plt.savefig(caminho_salvamento, dpi=300, bbox_inches='tight')
        plt.close(fig)
        
    # Criando resultados por municípios
    ipa_resultados_municipios = pd.pivot_table(
        df_formativa,
        index='municipio',           # O que vai ficar nas linhas
        columns=f'classificacao_ipa_f{NUM_FORMATIVA}', 
        values='cpf',                # O que vai ser contado (alunos)
        aggfunc='nunique',              # A operação: contar a quantidade de linhas (alunos)
        fill_value=0,                 # Preencher com 0 onde não houver alunos (em vez de NaN)
        observed=False
    ) # Reseta o índice para transformar 'municipio' em coluna normal

    # Adicionar uma coluna de 'Total' para enriquecer a tabela
    ipa_resultados_municipios['Total'] = ipa_resultados_municipios.sum(axis=1)

    # Ordenar a tabela pela quantidade total de alunos no município (do maior para o menor)
    ipa_resultados_municipios = ipa_resultados_municipios.sort_values(by='Total', ascending=False)

    # Salvando o arquivo no diretório dir_graficos
    nome_do_arquivo_4 = f'IPA_{PREFIXO_COLUNA}_resultados_por_municipio.xlsx'
    caminho_excel_4 = os.path.join(dir_graficos, nome_do_arquivo_4)
    ipa_resultados_municipios.to_excel(caminho_excel_4, index=True)

    print(f"Sucesso! Gráficos com espessura padronizada salvos em: {dir_graficos}")
    display(ipa_resultados_municipios.head())


## 4. Monitoramento de Metas de Enturmação (Conformidade com o TR)

O Termo de Referência do PBA SE 2026 estipula limites para a formação das turmas, visando garantir a qualidade pedagógica e a otimização dos recursos. 

**Parâmetros Analisados:**
* **Limite Mínimo:** 14 alunos (referência para zona urbana).
* **Limite Máximo:** 27 alunos por turma.

Nesta seção, avaliamos a média de alfabetizandos por turma em cada município parceiro. Municípios fora desta faixa exigirão justificativas técnicas (ex: turmas em áreas rurais de difícil acesso, que permitem mínimo de 7 alunos) ou readequação de enturmação.

In [ ]:
# Agrupamento e cálculo da média de alunos por turma por município
resumo_metas = df_turmas.groupby('municipio', observed=True).agg(
    total_turmas=('turma', 'count'),
    media_alunos_turma=('qtd_alfabetizandos', 'mean')
).reset_index()

# Ordenando do maior para o menor para facilitar a visualização
resumo_metas = resumo_metas.sort_values('media_alunos_turma', ascending=False)

# Configuração do Gráfico
fig, ax = plt.subplots(figsize=(18, 7))

# Usaremos a cor Azul (#005088) como padrão
sns.barplot(
    data=resumo_metas, 
    x='municipio', 
    y='media_alunos_turma', 
    color=cores_pba[0], 
    ax=ax
)

# Adicionando as Linhas de Meta do TR
ax.axhline(14, color=cores_pba[1], linestyle='--', linewidth=2.5, label='Mínimo Padrão (14)') # Amarelo
ax.axhline(27, color=cores_pba[2], linestyle='--', linewidth=2.5, label='Máximo Permitido (27)') # Verde

# Adicionando os Rótulos de Dados nas barras
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', padding=5, fontweight='bold', color='#333333', fontsize=9, rotation=90)

# Ajustes Estéticos
ax.set_title('Conformidade de Enturmação: Média de Alunos por Turma vs. Limites do TR', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Município', fontsize=12, fontweight='bold')
ax.set_ylabel('Média de Alunos por Turma', fontsize=12, fontweight='bold')
plt.xticks(rotation=90)

# Ajuste do limite Y para não cortar os rótulos
ax.set_ylim(0, resumo_metas['media_alunos_turma'].max() * 1.3)

ax.legend(loc='upper right', frameon=True)
sns.despine()
plt.tight_layout()

# Salvando o Gráfico
nome_grafico_8 = '8_metas_enturmacao_municipio.png'
caminho_grafico_8 = os.path.join(dir_graficos, nome_grafico_8)
plt.savefig(caminho_grafico_8, dpi=300, bbox_inches='tight')
plt.show()

print("Gráfico de metas de enturmação gerado e salvo com sucesso.")


## 5. Monitoramento de Frequência e Risco de Evasão

A retenção do alfabetizando é um dos maiores desafios da EJA (Educação de Jovens e Adultos). O monitoramento da frequência é o indicador antecedente primário para o risco de evasão.

**Critério de Análise:**
* O índice mínimo de aceitação padrão em programas educacionais do estado costuma ser de **75% de frequência**. 
* Abaixo desse limite, a coordenação local deve acionar a estratégia de busca ativa.

*Nota: A taxa é calculada dividindo o número de presenças computadas pela quantidade de aulas dadas na turma até o momento do snapshot.*

In [ ]:
# Tratamento e Cálculo da Taxa de Frequência
# Usamos np.where para evitar erro de divisão por zero caso haja turmas recém-criadas com 0 aulas dadas
df_pedagogico['taxa_frequencia'] = np.where(
    df_pedagogico['qtd_aulas_dadas_turma'] > 0,
    (df_pedagogico['qtd_presenca_alfabetizando'] / df_pedagogico['qtd_aulas_dadas_turma']) * 100,
    0
)

# Agrupamento por município para ver a frequência média
freq_muni = df_pedagogico.groupby('municipio', observed=True)['taxa_frequencia'].mean().reset_index()
freq_muni = freq_muni.sort_values('taxa_frequencia', ascending=False)

# Configuração do Gráfico
fig, ax = plt.subplots(figsize=(18, 7))

# Gráfico de barras na cor Azul institucional
sns.barplot(
    data=freq_muni, 
    x='municipio', 
    y='taxa_frequencia', 
    color=cores_pba[0], 
    ax=ax
)

# Linha de Alerta de Frequência (75%)
ax.axhline(75, color=cores_pba[1], linestyle='--', linewidth=2.5, label='Alerta de Busca Ativa (75%)') # Amarelo

# Adicionando os Rótulos de Dados
for container in ax.containers:
    # Usamos fmt='%.1f%%' para adicionar o símbolo de porcentagem no gráfico
    ax.bar_label(container, fmt='%.1f%%', padding=5, fontweight='bold', color='#333333', fontsize=9, rotation=90)

# Ajustes Estéticos
ax.set_title('Mapeamento de Engajamento: Frequência Média por Município', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Município', fontsize=12, fontweight='bold')
ax.set_ylabel('Taxa de Presença Média (%)', fontsize=12, fontweight='bold')
plt.xticks(rotation=90)

# Forçamos o eixo Y a ir até 115 para caber bem os rótulos de 100%
ax.set_ylim(0, 115) 

ax.legend(loc='lower left', frameon=True)
sns.despine()
plt.tight_layout()

# Salvando o Gráfico
nome_grafico_9 = ('9_frequencia_media_municipio.png')
caminho_grafico_9 = os.path.join(dir_graficos, nome_grafico_9)
plt.savefig(caminho_grafico_9, dpi=300, bbox_inches='tight')
plt.show()

# Extração Executiva: Contagem de Alunos Críticos
alunos_risco = df_pedagogico[df_pedagogico['taxa_frequencia'] < 75].shape[0]
percentual_risco = (alunos_risco / len(df_pedagogico)) * 100

print(f"ALERTA EXECUTIVO:")
print(f"Total de alunos com frequência abaixo de 75%: {alunos_risco} ({percentual_risco:.1f}% da amostra ativa).")
print("Recomenda-se exportar a lista nominal destes alfabetizandos para os coordenadores locais.")


## 6. Análise da Avaliação Socioemocional

Nesta seção, exploramos as respostas dadas ao questionário socioemocional aplicado (questões 1 a 9). O objetivo é entender o contexto de motivação, frequência e apoio familiar dos alfabetizandos.


In [ ]:
# Criando condição para avaliar atividade Socioemocional
if ATIVIDADE in [0, 5]:

    if ATIVIDADE == 0:
        PREFIXO_SOCIO_COLUNA = 'socio_entr'
        NOME_AVALIACAO_SOCIO = 'Socioemocional de Entrada'
        
    else:
        PREFIXO_SOCIO_COLUNA = 'socio_said'
        NOME_AVALIACAO_SOCIO = 'Socioemocional de Saída'

    # Construindo dicionário com o enunciado de cada questão da Socioemocional
    dicionario_questoes_socio = {
        'socio_entr_q1_resum': 'Q1 - Quanto à motivação para os estudos, \no(a) alfabetizando(a) apresenta:',
        'socio_entr_q2_resum': 'Q2 - Quanto à frequência e permanência nos encontros, \no(a) alfabetizando(a) apresenta:',
        'socio_entr_q3_resum': 'Q3 - Quanto à realização das atividades de leitura e escrita, \no(a) alfabetizando(a) apresenta:',
        'socio_entr_q4_resum': 'Q4 - Quanto à realização de novas atividades ou desafios, \no(a) alfabetizando(a) apresenta:',
        'socio_entr_q5_resum': 'Q5 - Quanto às dificuldades na leitura e escrita, \no(a) alfabetizando(a) apresenta:',
        'socio_entr_q6_resum': 'Q6 - Quanto às atividades em grupo, \no(a) alfabetizando(a) apresenta:',
        'socio_entr_q7_resum': 'Q7 - Quanto à utilidade da leitura e escrita no dia a dia, \no(a) alfabetizando(a) apresenta:', 
        'socio_entr_q8_resum': 'Q8 - Quais desafios foram observados \nnas primeiras semanas de aula?',
        'socio_said_q1_resum': 'Q1 - Em relação à motivação para estudar, apresentou:', 
        'socio_said_q2_resum': 'Q2 - Quanto à permanência e frequência:',
        'socio_said_q3_resum': 'Q3 - Na realização das atividades de leitura e escrita:',
        'socio_said_q4_resum': 'Q4 - Diante de novas atividades ou desafios:',
        'socio_said_q5_resum': 'Q5 - Quando surgiram erros ou dificuldades na leitura e escrita:',
        'socio_said_q6_resum': 'Q6 - Durante as atividades em grupo no decorrer do projeto:',
        'socio_said_q7_resum': 'Q7 - Reconheceu a utilidade da leitura e escrita para o dia a dia:',
        'socio_said_q8_resum': 'Q8 - Quanto à aquisição da leitura e escrita, apresentou resultados:',
        'socio_said_q9_resum': 'Q9 - Com relação aos desafios observados nas primeiras semanas de aula:'
    }

    # Configurando estilo dos gráficos
    sns.set_theme(style="whitegrid")

    # Definindo as colunas (vamos pegar da 1 a 9 para o loop automático)
    socio_cols = [f'{PREFIXO_SOCIO_COLUNA}_q{i}_resum' for i in range(1, 10)]

    # Entendendo número de respostas válidas para cada questão da socioemocional
    for col in socio_cols:
        if col in df_pedagogico.columns and df_pedagogico[col].notna().sum() > 0:
            print(f"Analisando a coluna: {col}")
            print(df_pedagogico[col].value_counts())
            print(f"Total de respospostas válidas: {df_pedagogico[col].count()}")
            print("-"*50)

    # 1. Filtrar dinamicamente as colunas de 1 a 9 que possuem dados
    cols_com_dados = [
        col for col in socio_cols 
        if col in df_pedagogico.columns and df_pedagogico[col].notna().sum() > 0
    ]

    # 2. Criando a grade fixa de 4 linhas e 2 colunas (8 espaços totais)
    fig = plt.figure(figsize=(16, 24)) # Aumentei um pouco a altura para as 5 linhas
    gs = gridspec.GridSpec(5, 2, figure=fig)

    # Criando uma lista para armazenar os eixos (axes) na ordem desejada
    axes = []

    # Preenchendo as 4 primeiras linhas (com 2 colunas cada) - Total: 8 gráficos
    for row in range(4):
        for col in range(2):
            axes.append(fig.add_subplot(gs[row, col]))

    # Adicionando o 9º gráfico na 5ª linha, ocupando ambas as colunas (gs[4, :])
    axes.append(fig.add_subplot(gs[4, :]))

    # 3. Loop para desenhar as questões
    resultados_socio = []

    for i, col in enumerate(cols_com_dados):
        # Proteção: garante que não vamos tentar desenhar mais gráficos do que os 9 espaços criados
        if i >= len(axes):
            break

        # 1. "Explodir" a coluna atual (separa listas em linhas individuais)
        df_plot = df_pedagogico.explode(col).copy()
        
        # 2. Se a coluna for categórica, adicionamos a categoria permitida antes de preencher
        if isinstance(df_plot[col].dtype, pd.CategoricalDtype):
            if 'Nenhuma / Sem resposta' not in df_plot[col].cat.categories:
                df_plot[col] = df_plot[col].cat.add_categories('Nenhuma / Sem resposta')
                
        # 3. Preencher listas vazias/nulos com o texto padrão
        df_plot[col] = df_plot[col].fillna('Nenhuma / Sem resposta')
        
        # 4. Gerar o gráfico usando o novo df_plot
        sns.countplot(
            y=col, 
            data=df_plot, 
            order=df_plot[col].value_counts().index, 
            palette='viridis', 
            hue=col,            
            legend=False,       
            ax=axes[i]
        )
        
        # Ajustando o Título
        titulo = dicionario_questoes_socio.get(col, col)
        axes[i].set_title(titulo, fontsize=16, fontweight='bold', pad=15)
        
        # Ajustando Eixos
        axes[i].set_xlabel('Quantidade de Alfabetizandos', fontsize=14)
        axes[i].set_ylabel('', fontsize=14)
        
        # Quebrando o texto longo do eixo Y em múltiplas linhas (max 35 caracteres por linha)
        labels_atuais = [label.get_text() for label in axes[i].get_yticklabels()]
        labels_quebradas = [textwrap.fill(texto, width=35) for texto in labels_atuais]
        
        axes[i].set_yticks(range(len(labels_quebradas))) # Evita warnings do matplotlib
        axes[i].set_yticklabels(labels_quebradas, fontsize=12)
        axes[i].tick_params(axis='x', labelsize=14)
        
        # Calcula as contagens da coluna atual e transforma em um DataFrame
        contagem = df_plot[col].value_counts().reset_index()
        contagem.columns = ['Resposta', 'Quantidade']
        
        # Cria uma coluna para registrar de qual questão vieram essas respostas
        contagem['Pergunta'] = col  
        
        # Guarda o DataFrame desta questão na lista
        resultados_socio.append(contagem)

    # Junta todos os resultados em um único DataFrame
    df_socio_dados = pd.concat(resultados_socio, ignore_index=True)

    # Salvando dados socioemocionais em Excel
    nome_socio_dados = f'dados_socioemocionais_{PREFIXO_SOCIO_COLUNA}.xlsx'
    caminho_dados_socio = os.path.join(dir_graficos, nome_socio_dados)
    df_socio_dados.to_excel(caminho_dados_socio, index=False)

    # 4. Limpeza e ajustes finais
    # Remove eixos que ficaram vazios (caso cols_com_dados tenha menos de 9 itens válidos)
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    # Ajusta o espaçamento entre os gráficos (w_pad aumenta o espaço entre as colunas)
    plt.tight_layout(pad=2.0, w_pad=6.0, h_pad=3.0)
        
    # IMPORTANTE: Salvar o gráfico DEVE vir antes do plt.show()
    nome_grafico_10 = '10_analise_socioemocional.png'
    caminho_grafico_10 = os.path.join(dir_graficos, nome_grafico_10)
    plt.savefig(caminho_grafico_10, dpi=600, bbox_inches='tight')

    # Exibe na tela após salvar
    plt.show()
    
else:
    print("Dados de Avaliação Socioemocional são relevantes somente nas avaliações de Entrada e de Saída.")


# 7. Distribuição Espacial da Aprendizagem (Mapa de Calor)

Para refinar o planejamento estratégico, este item apresenta uma visualização geoespacial (mapa coroplético) focada nos alfabetizandos que se encontram nos níveis mais altos do diagnóstico de entrada (**N3 e N4**).

Municípios com cores mais quentes indicam uma alta proporção de alunos que já possuem habilidades de leitura silábica ou silábico-alfabética. Isso sinaliza para a coordenação pedagógica que tais regiões podem necessitar de acervos literários mais avançados, enquanto os municípios em cores claras necessitam de forte intervenção fonológica.

> **Metodologia:** Calculamos o percentual de alunos N3 e N4 em relação ao total de alunos ativos de cada município e cruzamos com a malha territorial (Shapefile) do estado de Sergipe.

> **Observação:** O mapa de calor só será gerado se estivermos analisando o diagnóstico de entrada.

In [ ]:
# Checagem se avaliação analisada é  Atividade Diagnóstica
if ATIVIDADE != 0:
    print("Não é possível gerar o gráfico de Mapa de Calor para a Avaliação Analisada.")

else:
    # Funções de tratamento de texto
    def padronizar_nomes(texto):
        return (
            texto.str.upper()
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.replace(r"\b(DE|DA|DO|DAS|DOS)\b", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    def abreviar_nome(nome):
        substituicoes = {
            "NOSSA SENHORA": "NS",
            "N.S.": "NS",
            "TOBIAS BARRETO": "T. BARRETO",
            "RIACHÃO DO DANTAS": "R. DANTAS",
            "SANTA LUZIA DO ITANHY": "ST LUZIA ITANHY", # Removido o 'ª' para evitar erro de encoding no plot
            "SAO FRANCISCO": "S. FRANCISCO"
        }
        nome = str(nome).upper()
        for original, abreviado in substituicoes.items():
            nome = nome.replace(original, abreviado)
        return nome.title()

    # PREPARAÇÃO DOS DADOS PEDAGÓGICOS
    # A. Total de alunos ativos por município
    total_muni = df_pedagogico.groupby('municipio', observed=True)['cpf'].count().reset_index()
    total_muni.columns = ['municipio', 'total_alunos']

    # B. Filtrar apenas alunos N3 e N4 e contar
    df_n3_n4 = df_pedagogico[df_pedagogico['diag_entr_result_resum'].isin(['N3', 'N4'])]
    n3_n4_muni = df_n3_n4.groupby('municipio', observed=True)['cpf'].count().reset_index()
    n3_n4_muni.columns = ['municipio', 'total_n3_n4']

    # C. Juntar e calcular o percentual
    df_mapa_dados = pd.merge(total_muni, n3_n4_muni, on='municipio', how='left').fillna(0)
    df_mapa_dados['pct_n3_n4'] = (df_mapa_dados['total_n3_n4'] / df_mapa_dados['total_alunos']) * 100

    # Aplicar a função de padronização no dataset pedagógico
    df_mapa_dados['nome_padrao'] = padronizar_nomes(df_mapa_dados['municipio'])

    # 3. CARREGAMENTO E JOIN ESPACIAL (SHAPEFILE)
    caminho_shp = 'data_files/SE_Municipios_2024/SE_Municipios_2024.shp'
    gdf_se = gpd.read_file(caminho_shp)

    # ATENÇÃO: Os shapefiles do IBGE geralmente usam 'NM_MUN' como nome da coluna do município.
    # Se o seu shapefile tiver outro nome (ex: 'NOME', 'MUNICÍPIO'), altere a linha abaixo!
    coluna_nome_shapefile = 'NM_MUN'

    # Padronizar nomes no Shapefile
    gdf_se['nome_padrao'] = padronizar_nomes(gdf_se[coluna_nome_shapefile])

    # Fazer o Join Espacial (Merge do Shapefile com os dados do programa)
    gdf_mapa = gdf_se.merge(df_mapa_dados, on='nome_padrao', how='left')

    # Preencher municípios que não participam do programa (NaN) com -1 para pintá-los de cinza
    gdf_mapa['pct_n3_n4'] = gdf_mapa['pct_n3_n4'].fillna(-1)

    # Aplicar abreviação para os rótulos do mapa
    gdf_mapa['nome_label'] = gdf_mapa[coluna_nome_shapefile].apply(abreviar_nome)

    # 4. PLOTAGEM DO MAPA DE CALOR
    fig, ax = plt.subplots(1, 1, figsize=(16, 16))

    # A. Criar o Colormap (Ajuste de Saturação para o Verde)
    # Começamos com um Verde Sálvia denso (#99CCB1) para evitar o aspecto pálido
    # e terminamos no Verde Institucional PBA (#00843D)
    cores_gradiente = ['#99CCB1', '#00843D'] 
    cmap_verde_pba = LinearSegmentedColormap.from_list('Verde_PBA_Forte', cores_gradiente)

    # B. Plotar municípios INATIVOS (Cinza de contraste)
    gdf_mapa[gdf_mapa['pct_n3_n4'] == -1].plot(
        ax=ax, color='#D4D4D4', edgecolor='white', linewidth=0.8
    )

    # C. Definir limite de contraste (Cortando outliers)
    dados_ativos = gdf_mapa[gdf_mapa['pct_n3_n4'] >= 0]
    limite_maximo_cor = dados_ativos['pct_n3_n4'].quantile(0.95)

    # D. Plotar municípios ATIVOS (Com o degradê verde)
    mapa_plot = dados_ativos.plot(
        column='pct_n3_n4',
        cmap=cmap_verde_pba,  # Aplicando a nova paleta verde
        ax=ax,
        edgecolor='white',
        linewidth=1.0, 
        vmax=limite_maximo_cor, 
        legend=True,
        legend_kwds={
            'label': "Proporção de Alunos N3 e N4 (%)", 
            'orientation': "horizontal", 
            'shrink': 0.5,
            'pad': 0.02
        }
    )

    # E. Adicionar rótulos
    for idx, row in dados_ativos.iterrows():
        centroide = row.geometry.centroid
        
        ax.annotate(
            text=row['nome_label'], 
            xy=(centroide.x, centroide.y), 
            ha='center', va='center',
            fontsize=7, color='#1A1A1A', fontweight='bold', 
            bbox=dict(facecolor='white', alpha=0.85, edgecolor='#CCCCCC', linewidth=0.5, boxstyle='round,pad=0.2')
        )

    # F. Estética final
    ax.set_title('Concentração Regional: Proporção de Alunos Níveis 3 e 4 (PBA SE 2026)', 
                fontsize=18, fontweight='bold', pad=20) # Título também no verde oficial
    ax.axis('off')

    # Salvar
    nome_grafico_11 = '11_mapa_calor_n3_n4.png'
    caminho_grafico_11 = os.path.join(dir_graficos, nome_grafico_11)
    plt.savefig(caminho_grafico_11, dpi=300, bbox_inches='tight')
    plt.show()

    print("Mapa de Calor gerado e salvo com sucesso!")